## Notebook grammar
`setup → config → construct → conditions → simulate → epoch → analyze → report`

Open in Colab: https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v040_continuous_omission_oddball.ipynb

# v0.4.0 · Continuous Sequential Omission Oddball Paradigm

**Biological question:** How does a sensory omission — the expected-but-absent  
item in a 4-slot sequential stream — modulate population spiking and laminar  
proxy fields in a 100-neuron V1 column with active homeostasis and plasticity?

**Paradigm:** 4-item sequences with L4 E neurons driven at each stimulus slot.

| Condition | Sequence | Omission position |
|---|---|---|
| AAAA | A · A · A · A | none (70% common) |
| AAAX | A · A · A · X | p4 |
| AAXA | A · A · X · A | p3 |
| AXAA | A · X · A · A | p2 |

**Scope:** proxy scaffold — not calibrated physics.


In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,optax]"])


In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd, json
from pathlib import Path
from types import SimpleNamespace
import jaxfne as jtfne
from jaxfne.tutorial_utils import spectrolaminar_from_trials

jtfne.enable_x64()
print(f"jaxfne {jtfne.__version__}   jax {jax.__version__}")
print("Devices:", jax.devices())


## Scope — Computational Scaffold & Truth Gates

| Gate | Value |
|---|---|
| claim_level | computational_scaffold |
| field_solver_status | linear_solver |
| physical_amplitude_calibrated | False |

Outputs tagged `*_proxy`. No physical-amplitude or mechanism claims.


## 1 · Paradigm timing

Canonical timing from the sequential visual omission paradigm  
(fixation baseline + 4 stimulus slots + 4 delay gaps).

| Epoch | Onset (ms) | Duration (ms) |
|---|---:|---:|
| fx (fixation) | −500 | 500 |
| p1 | 0 | 531 |
| d1 | 531 | 500 |
| p2 | 1031 | 531 |
| d2 | 1562 | 500 |
| p3 | 2062 | 531 |
| d3 | 2593 | 500 |
| p4 | 3093 | 531 |
| d4 | 3624 | 500 |

p1-relative time 0 ms = onset of the first stimulus.  
Omission-relative time 0 ms = expected onset of the missing stimulus.


In [ ]:
# ── Editable parameters ────────────────────────────────────────────────────────
N_NEURONS     = 100          # neurons in V1 column
N_TRIALS      = 10           # independent trials per condition
DT_MS         = 0.1          # timestep (ms)
SEED          = 42
N_CONTACTS    = 16
FREQ_MIN_HZ   = 1.0; FREQ_MAX_HZ = 150.0; FREQ_COUNT = 64
AB_HZ         = (10.0, 25.0)
GA_HZ         = (40.0, 150.0)

# Stimulus parameters
A_AMP         = 5.0   # native Izhikevich current (uncalibrated)
FX_MS         = 500.0 # fixation/pre-sequence baseline

# Canonical slot timing (p1-relative, ms)
P_DUR_MS      = 531.0  # stimulus presentation duration
D_DUR_MS      = 500.0  # inter-stimulus delay duration

P1_ONSET_MS   = 0.0
D1_ONSET_MS   = P1_ONSET_MS + P_DUR_MS          # 531
P2_ONSET_MS   = D1_ONSET_MS + D_DUR_MS          # 1031
D2_ONSET_MS   = P2_ONSET_MS + P_DUR_MS          # 1562
P3_ONSET_MS   = D2_ONSET_MS + D_DUR_MS          # 2062
D3_ONSET_MS   = P3_ONSET_MS + P_DUR_MS          # 2593
P4_ONSET_MS   = D3_ONSET_MS + D_DUR_MS          # 3093
SEQ_END_MS    = P4_ONSET_MS + P_DUR_MS + D_DUR_MS  # 4124

# Full trial: fixation + sequence
TOTAL_MS      = FX_MS + SEQ_END_MS              # 4624 ms
n_steps       = int(TOTAL_MS / DT_MS)           # 46240

# Condition distribution: AAAA=70%, each X=10%
CONDITIONS    = ["AAAA", "AAAX", "AAXA", "AXAA"]
COND_PROBS    = [0.70,   0.10,   0.10,   0.10]

# Homeostasis + plasticity params (active during all simulations)
HOMEO_PARAMS  = dict(r_star=0.05, tau_r_ms=300.0, alpha=1.0, k_gain=1.0,
                      g_min=-12.0, g_max=8.0, r_max=1.0, eta=0.01,
                      tau_x_ms=100.0, w_min=-10.0, w_max=10.0)

OUTPUT_DIR    = Path("outputs/v040_omission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timing in trial-absolute ms (fixation offsets all slots by FX_MS)
slots_ms = {
    "p1": FX_MS + P1_ONSET_MS,
    "p2": FX_MS + P2_ONSET_MS,
    "p3": FX_MS + P3_ONSET_MS,
    "p4": FX_MS + P4_ONSET_MS,
}
omission_onset_ms = {cond: None for cond in CONDITIONS}
for cond, slot in [("AAAX","p4"),("AAXA","p3"),("AXAA","p2")]:
    omission_onset_ms[cond] = slots_ms[slot]

print(f"Full trial: {TOTAL_MS:.0f} ms   n_steps={n_steps}")
print(f"Slot onsets (trial-abs): { {k:f'{v:.0f}ms' for k,v in slots_ms.items()} }")
print(f"Homeostasis+plasticity: {HOMEO_PARAMS}")


## 2 · Build canonical V1 column

100-neuron column, edge-list backend, with homeostasis+plasticity active.  
L4 E neurons are the paradigm stimulus targets.


In [ ]:
cfg = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "LFP-proxy", "CSD-proxy"], n_contacts=N_CONTACTS)
    .field(domain="laminar_column", conductivity="proxy",
           boundary="mean_zero_neumann")
)
model = jtfne.construct(cfg)
nt     = model.neuron_table()
N_TOT  = len(nt)

# L4 E neuron indices — paradigm stimulus targets
l4e_idx = [i for i, r in enumerate(nt)
            if r.get("layer") in ("L4",) and r.get("cell_type") == "E"]
print(f"Column: {N_TOT} neurons   L4 E (stimulus targets): {len(l4e_idx)}")

# Cell-type distribution
from collections import Counter
ct_counts = Counter(r["cell_type"] for r in nt)
lay_counts = Counter(r["layer"] for r in nt)
print("Cell types:", dict(ct_counts))
print("Layers:    ", dict(lay_counts))


## 3 · Drive schedule builders per condition

Each condition maps to a `StimulusSchedule` selecting which slots carry drive.  
Omission slot → no event (zero drive at that time-point).


In [ ]:
def _slot_event(slot_name: str, onset_ms_trial: float, active: bool = True) -> dict:
    """Build one slot event dict for StimulusSchedule."""
    return {
        "onset_ms":     onset_ms_trial,
        "duration_ms":  P_DUR_MS,
        "amplitude":    A_AMP if active else 0.0,
        "label":        slot_name if active else f"X({slot_name})",
        "is_drive_event": active,
        "target_indices": l4e_idx,
    }

# Condition → which slots are stimulated (True) vs omitted (False)
COND_SLOT_MAP = {
    "AAAA": {"p1": True,  "p2": True,  "p3": True,  "p4": True},
    "AAAX": {"p1": True,  "p2": True,  "p3": True,  "p4": False},
    "AAXA": {"p1": True,  "p2": True,  "p3": False, "p4": True},
    "AXAA": {"p1": True,  "p2": False, "p3": True,  "p4": True},
}

def make_schedule(cond: str) -> jtfne.StimulusSchedule:
    slot_map = COND_SLOT_MAP[cond]
    events = tuple(
        _slot_event(s, slots_ms[s], active=slot_map[s])
        for s in ("p1","p2","p3","p4")
        if slot_map[s]          # only add active (non-omission) events
    )
    return jtfne.StimulusSchedule(events=events, n_neurons=N_TOT)

schedules = {cond: make_schedule(cond) for cond in CONDITIONS}
for cond, sched in schedules.items():
    n_ev = len(sched.events)
    print(f"  {cond}: {n_ev} drive event(s)   "
          f"{'[omission at ' + ('p2' if cond=='AXAA' else 'p3' if cond=='AAXA' else 'p4') + ']' if cond != 'AAAA' else '[no omission]'}")


## 4 · Simulate: 10 trials × 4 conditions

Homeostasis (`k_gain=1.0`, `eta=0.01` plasticity) active throughout.  
JIT compiles on the first trial; remaining 39 trials reuse the compiled kernel.


In [ ]:
runtime_cfg = jtfne.RuntimeConfig(
    recurrent_backend="edge_list",
    enable_homeostasis=True,
    homeostasis_params=HOMEO_PARAMS,
)

results = {}  # cond -> {"spikes": (T,N), "csd": (T,nc), "lfp": (T,nc), per trial list}
print(f"Simulating {len(CONDITIONS)} conditions × {N_TRIALS} trials × {TOTAL_MS:.0f} ms ...")
print(f"  (n_steps={n_steps}  —  JIT warmup on first trial)")

for cond in CONDITIONS:
    sched = schedules[cond]
    csd_trials, lfp_trials, spk_trials = [], [], []
    for i in range(N_TRIALS):
        sig = jtfne.simulate(
            model,
            sim=jtfne.Simulation(duration_ms=TOTAL_MS, dt_ms=DT_MS,
                                   seed=SEED + i, runtime=runtime_cfg),
            paradigm=sched,
        )
        csd_trials.append(np.asarray(sig.field.csd_proxy))   # (T, n_contacts)
        lfp_trials.append(np.asarray(sig.field.lfp_proxy))
        spk_trials.append(np.asarray(sig.spikes))             # (T, N)
    results[cond] = {
        "csd": np.stack(csd_trials),  # (N_TRIALS, T, n_contacts)
        "lfp": np.stack(lfp_trials),
        "spk": np.stack(spk_trials),  # (N_TRIALS, T, N)
    }
    mean_rate = float(results[cond]["spk"].mean() * 1000.0 / DT_MS)
    print(f"  {cond}: done   mean_rate≈{mean_rate:.1f} Hz")

print("\nAll conditions simulated.")


## 5 · Population firing rates per epoch per condition

In [ ]:
time_ms = np.arange(n_steps) * DT_MS

def epoch_mask(onset_ms, dur_ms):
    return (time_ms >= onset_ms) & (time_ms < onset_ms + dur_ms)

# Epochs: fixation, each slot, each delay
epoch_defs = {
    "fix":    epoch_mask(0, FX_MS),
    "p1":     epoch_mask(slots_ms["p1"], P_DUR_MS),
    "d1":     epoch_mask(slots_ms["p1"]+P_DUR_MS, D_DUR_MS),
    "p2":     epoch_mask(slots_ms["p2"], P_DUR_MS),
    "d2":     epoch_mask(slots_ms["p2"]+P_DUR_MS, D_DUR_MS),
    "p3":     epoch_mask(slots_ms["p3"], P_DUR_MS),
    "d3":     epoch_mask(slots_ms["p3"]+P_DUR_MS, D_DUR_MS),
    "p4":     epoch_mask(slots_ms["p4"], P_DUR_MS),
    "d4":     epoch_mask(slots_ms["p4"]+P_DUR_MS, D_DUR_MS),
}

rate_rows = []
for cond in CONDITIONS:
    spk = results[cond]["spk"]  # (N_TRIALS, T, N)
    for ep, mask in epoch_defs.items():
        dur_s = mask.sum() * DT_MS / 1000.0
        rate = float(spk[:, mask, :].mean() * 1000.0 / DT_MS) if dur_s > 0 else 0.0
        rate_rows.append({"condition": cond, "epoch": ep, "mean_rate_hz": round(rate, 2)})

df_rates = pd.DataFrame(rate_rows).pivot(index="epoch", columns="condition",
                                           values="mean_rate_hz")
df_rates = df_rates.reindex(list(epoch_defs.keys()))
print("Mean population firing rate (Hz) per epoch and condition:")
print(df_rates.to_string(float_format=lambda x: f"{x:6.2f}"))


## 6 · Spike rasters per condition (trial 0)

In [ ]:
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(14, 4), sharey=True, constrained_layout=True)
for ax, cond in zip(axes, CONDITIONS):
    spk0 = results[cond]["spk"][0]  # (T, N)
    t_spk, n_spk = np.where(spk0)
    ax.scatter(time_ms[t_spk], n_spk, s=0.8, c="black", alpha=0.6, rasterized=True)
    # Shade stimulus slots
    slot_colors = {"p1":"#4477AA33","p2":"#EE665533","p3":"#66BB6633","p4":"#AA44BB33"}
    for sn, scol in slot_colors.items():
        if COND_SLOT_MAP[cond][sn]:
            ax.axvspan(slots_ms[sn], slots_ms[sn]+P_DUR_MS, color=scol, zorder=0)
        else:
            ax.axvspan(slots_ms[sn], slots_ms[sn]+P_DUR_MS, color="#FFDDDD44",
                        hatch="//", zorder=0, label="omission")
    ax.set_title(f"{cond} — proxy raster", fontsize=9)
    ax.set_xlabel("time (ms)"); ax.set_xlim(0, TOTAL_MS)
axes[0].set_ylabel("neuron idx")
fig.suptitle("Spike rasters — proxy scaffold (trial 0 per condition)", fontsize=10)
fig.savefig(OUTPUT_DIR / "rasters.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved: rasters.png  (proxy scaffold)")


## 7 · Local omission-locked LFP-proxy

Epoch: −500 to +800 ms around the expected omission onset (time 0 = omission).  
Late pre-omission baseline: −250 to −50 ms.  
AAAA is aligned to the same slot for direct comparison.


In [ ]:
WIN_PRE_MS  = 500.0   # ms before omission onset
WIN_POST_MS = 800.0   # ms after
BL_PRE_MS   = 250.0   # baseline start before omission
BL_POST_MS  = 50.0    # baseline end before omission

# For AAAA, align to p4 onset for a matched non-omission reference
ALIGN_MS = {
    "AAAA": slots_ms["p4"],
    "AAAX": slots_ms["p4"],
    "AAXA": slots_ms["p3"],
    "AXAA": slots_ms["p2"],
}

local_windows = {}

for cond in CONDITIONS:
    align = ALIGN_MS[cond]
    t0_idx = int((align - WIN_PRE_MS) / DT_MS)
    t1_idx = int((align + WIN_POST_MS) / DT_MS)
    t0_idx = max(0, t0_idx); t1_idx = min(n_steps, t1_idx)
    win_len = t1_idx - t0_idx

    lfp = results[cond]["lfp"][:, t0_idx:t1_idx, :]  # (N_TRIALS, win_len, n_contacts)
    t_local_ms = np.arange(win_len) * DT_MS - WIN_PRE_MS  # 0=omission onset

    # Baseline correction: subtract mean over [-BL_PRE_MS, -BL_POST_MS]
    bl_start = int((WIN_PRE_MS - BL_PRE_MS) / DT_MS)
    bl_end   = int((WIN_PRE_MS - BL_POST_MS) / DT_MS)
    bl_start = max(0, bl_start); bl_end = min(win_len, bl_end)
    bl_mean  = lfp[:, bl_start:bl_end, :].mean(axis=1, keepdims=True)
    lfp_bl   = lfp - bl_mean

    local_windows[cond] = {
        "lfp_bl":   lfp_bl,   # (N_TRIALS, win_len, n_contacts)
        "t_ms":     t_local_ms,
        "align_ms": align,
    }
    print(f"  {cond}: aligned to {align:.0f} ms  window=[{-WIN_PRE_MS:.0f}, +{WIN_POST_MS:.0f}] ms"
          f"  shape={lfp_bl.shape}")


In [ ]:
# Plot mean LFP-proxy (trial average) across contacts for each condition
fig, axes = plt.subplots(len(CONDITIONS), 1, figsize=(10, 2.8*len(CONDITIONS)),
                          constrained_layout=True, sharex=True)
contact_colors = plt.cm.coolwarm(np.linspace(0, 1, N_CONTACTS))

for ax, cond in zip(axes, CONDITIONS):
    win = local_windows[cond]
    lfp_mean = win["lfp_bl"].mean(axis=0)  # (win_len, n_contacts)
    t_ms = win["t_ms"]
    for ci in range(N_CONTACTS):
        ax.plot(t_ms, lfp_mean[:, ci], color=contact_colors[ci], lw=0.6, alpha=0.7)
    ax.axvline(0, color="red", lw=1.2, ls="--", label="omission onset" if cond!="AAAA" else "p4 onset")
    ax.axvspan(-BL_PRE_MS, -BL_POST_MS, color="gray", alpha=0.12, label="baseline")
    ax.set_title(f"{cond} — LFP-proxy (trial-mean, baseline-corrected)  [proxy scaffold]",
                  fontsize=9)
    ax.set_ylabel("LFP-proxy (a.u.)")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.25)

axes[-1].set_xlabel("time from expected stimulus onset (ms)")
fig.suptitle("Local omission-locked LFP-proxy per condition\n"
              f"N={N_NEURONS} neurons · {N_TRIALS} trials · proxy scaffold",
              fontsize=10)
fig.savefig(OUTPUT_DIR / "local_omission_lfp.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved: local_omission_lfp.png  (proxy scaffold)")


## 8 · Spectrolaminar motif similarity per condition

Spectrolaminar profile (alpha-beta vs gamma depth cross) from the full trial CSD-proxy.  
Similarity = anti-correlation of the two depth profiles → 100% = ideal motif.


In [ ]:
col_h = 1.6e-3  # 1.6 mm column
n_c   = N_CONTACTS

cfg_ns = SimpleNamespace(
    dt_ms=DT_MS, duration_ms=TOTAL_MS, freq_min_hz=FREQ_MIN_HZ,
    freq_max_hz=FREQ_MAX_HZ, freq_count=FREQ_COUNT, l4_ref_rel=0.5,
    cz_m=col_h, output_dir=None, n_trials=N_TRIALS, areas=("V1",),
)

sim_scores = {}
specs_all  = {}

for cond in CONDITIONS:
    csd = results[cond]["csd"]  # (N_TRIALS, T, n_contacts)
    td  = {"csd_contacts": csd,
            "contact_depths_m": np.linspace(0.0, col_h, n_c)}
    _, spec = spectrolaminar_from_trials(
        td, cfg_ns, signal_key="csd_contacts",
        freq_min_hz=FREQ_MIN_HZ, freq_max_hz=FREQ_MAX_HZ, freq_count=FREQ_COUNT,
        alpha_beta_range_hz=AB_HZ, gamma_range_hz=GA_HZ,
        area_index=0, area_name="V1",
    )
    ab   = np.asarray(spec["alpha_beta"], np.float64)
    gm   = np.asarray(spec["gamma"], np.float64)
    corr = float(np.corrcoef(ab, gm)[0,1]) if ab.size>=2 and np.std(ab)>1e-9 and np.std(gm)>1e-9 else 0.0
    sim  = float(np.clip((1 - corr)*50, 0, 100))
    sim_scores[cond] = sim
    spec["similarity_pct"] = sim
    specs_all[cond] = spec
    print(f"  {cond}: similarity={sim:.1f}%  (ab-gm corr={corr:.3f})")


In [ ]:
fig, axes = plt.subplots(2, len(CONDITIONS), figsize=(14, 6), constrained_layout=True)
pos = np.linspace(-0.5*col_h, 0.5*col_h, n_c) * 1e3  # mm from L4

for ci, cond in enumerate(CONDITIONS):
    spec = specs_all[cond]
    ab   = spec["alpha_beta"]
    gm   = spec["gamma"]
    pwr  = spec["relative_power"]          # (freq_count, n_contacts)
    freqs = spec["freq_hz"]

    # Panel A: alpha-beta/gamma depth profiles
    ax = axes[0, ci]
    ax.plot(ab, pos, "b-o", ms=3, lw=1.5, label="alpha-beta (10-25 Hz)")
    ax.plot(gm, pos, "r-o", ms=3, lw=1.5, label="gamma (40-150 Hz)")
    ax.axhline(0, color="gray", lw=0.8, ls="--", label="L4 ref")
    ax.set_xlabel("norm. power"); ax.set_ylabel("depth from L4 (mm)")
    ax.set_title(f"{cond}\nsim={sim_scores[cond]:.1f}%", fontsize=9)
    ax.legend(fontsize=6); ax.grid(True, alpha=0.25)

    # Panel B: power heatmap (freq × depth)
    ax = axes[1, ci]
    im = ax.imshow(pwr, aspect="auto", cmap="plasma", origin="lower",
                    extent=[pos[0], pos[-1], freqs[0], freqs[-1]])
    ax.set_xlabel("depth from L4 (mm)"); ax.set_ylabel("freq (Hz)")
    ax.set_title(f"{cond} — CSD power (proxy)", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.04, label="rel.power")

fig.suptitle("Spectrolaminar motif per condition — proxy scaffold\n"
              f"N={N_NEURONS} neurons · {N_TRIALS} trials · homeostasis+plasticity active",
              fontsize=10)
fig.savefig(OUTPUT_DIR / "spectrolaminar_per_condition.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved: spectrolaminar_per_condition.png  (proxy scaffold)")


## 9 · Continuous sequential paradigm simulation

Simulate a continuous block of N_SEQ sequences drawn from the condition distribution  
(AAAA 70%, AAAX/AAXA/AXAA 10% each) as a single long simulation.  
Epoch-lock to omission slots post-hoc.


In [ ]:
N_SEQ = 20   # number of sequences in the continuous block

rng_seq = np.random.default_rng(SEED + 9999)
seq_conds = rng_seq.choice(CONDITIONS, size=N_SEQ,
                             p=COND_PROBS, replace=True)
print(f"Continuous block: {N_SEQ} sequences   distribution:")
for c in CONDITIONS:
    print(f"  {c}: {(seq_conds==c).sum()} / {N_SEQ}")

# Build one long drive schedule
total_cont_ms = N_SEQ * TOTAL_MS
n_cont_steps  = int(total_cont_ms / DT_MS)
cont_schedule  = np.zeros((n_cont_steps, N_TOT), dtype=np.float32)

omission_onsets_cont = []  # (seq_idx, cond, abs_time_ms)

for seq_i, cond in enumerate(seq_conds):
    seq_offset_ms = seq_i * TOTAL_MS
    slot_map = COND_SLOT_MAP[cond]
    for sn in ("p1","p2","p3","p4"):
        if not slot_map[sn]:
            omission_onsets_cont.append((seq_i, cond, seq_offset_ms + slots_ms[sn]))
            continue
        t_on  = seq_offset_ms + slots_ms[sn]
        t_off = t_on + P_DUR_MS
        i_on  = int(t_on / DT_MS)
        i_off = int(t_off / DT_MS)
        i_on  = max(0, min(i_on, n_cont_steps))
        i_off = max(0, min(i_off, n_cont_steps))
        cont_schedule[i_on:i_off, l4e_idx] += A_AMP

cont_sched_j = jtfne.StimulusSchedule(
    events=({"onset_ms": 0, "duration_ms": total_cont_ms, "amplitude": 0.0,
              "label": "placeholder", "is_drive_event": False},),
    n_neurons=N_TOT,
)
# Use drive_array directly via the schedule to_array mechanism — here we bake
# the schedule into a manual drive array injected via the model's _simulate_arrays.
# Workaround: pass as Simulation.poisson_drive is not applicable.
# Instead, materialise the schedule in events covering each active slot globally.
# Rebuild as true event list for the scheduler.
cont_events = []
for seq_i, cond in enumerate(seq_conds):
    seq_offset_ms = seq_i * TOTAL_MS
    slot_map = COND_SLOT_MAP[cond]
    for sn in ("p1","p2","p3","p4"):
        if slot_map[sn]:
            cont_events.append({
                "onset_ms": seq_offset_ms + slots_ms[sn],
                "duration_ms": P_DUR_MS, "amplitude": A_AMP,
                "label": f"seq{seq_i}_{cond}_{sn}",
                "is_drive_event": True, "target_indices": l4e_idx,
            })

cont_paradigm = jtfne.StimulusSchedule(
    events=tuple(cont_events), n_neurons=N_TOT,
)
print(f"Continuous schedule: {len(cont_events)} drive events  "
      f"total_dur={total_cont_ms:.0f} ms  omissions={len(omission_onsets_cont)}")


In [ ]:
print(f"Simulating continuous block ({total_cont_ms:.0f} ms = {n_cont_steps} steps) ...")
print("  Note: one trial — may take 1-3 min on CPU with JIT.")
sig_cont = jtfne.simulate(
    model,
    sim=jtfne.Simulation(duration_ms=total_cont_ms, dt_ms=DT_MS,
                           seed=SEED + 77, runtime=runtime_cfg),
    paradigm=cont_paradigm,
)
csd_cont = np.asarray(sig_cont.field.csd_proxy)  # (T_total, n_contacts)
spk_cont = np.asarray(sig_cont.spikes)            # (T_total, N)
time_cont = np.arange(n_cont_steps) * DT_MS
mean_rate_cont = float(spk_cont.mean() * 1000.0 / DT_MS)
print(f"Done.  Mean population rate = {mean_rate_cont:.1f} Hz")
print(f"finite CSD: {np.isfinite(csd_cont).all()}  "
      f"finite SPK: {np.isfinite(spk_cont).all()}")


## 10 · Epoch-lock omission events from continuous run

Extract omission-locked CSD-proxy windows (−500 to +800 ms)  
from the continuous simulation, grouped by condition.


In [ ]:
WIN_PRE_CT  = int(WIN_PRE_MS  / DT_MS)
WIN_POST_CT = int(WIN_POST_MS / DT_MS)
WIN_LEN_CT  = WIN_PRE_CT + WIN_POST_CT
t_local_ct  = np.arange(WIN_LEN_CT) * DT_MS - WIN_PRE_MS

omission_epochs = {cond: [] for cond in CONDITIONS if cond != "AAAA"}
aaaa_p4_epochs  = []   # matched non-omission reference

for seq_i, cond in enumerate(seq_conds):
    seq_offset_ms = seq_i * TOTAL_MS
    if cond == "AAAA":
        align_ms = seq_offset_ms + slots_ms["p4"]
    else:
        # omitted slot
        x_slot = {"AAAX":"p4","AAXA":"p3","AXAA":"p2"}[cond]
        align_ms = seq_offset_ms + slots_ms[x_slot]

    t0 = int((align_ms - WIN_PRE_MS) / DT_MS)
    t1 = t0 + WIN_LEN_CT
    if t0 < 0 or t1 > n_cont_steps:
        continue

    epoch_csd = csd_cont[t0:t1, :]  # (WIN_LEN_CT, n_contacts)
    # Baseline correct
    bl_s = int((WIN_PRE_MS - BL_PRE_MS) / DT_MS)
    bl_e = int((WIN_PRE_MS - BL_POST_MS) / DT_MS)
    bl_s = max(0, bl_s); bl_e = min(WIN_LEN_CT, bl_e)
    bl   = epoch_csd[bl_s:bl_e, :].mean(axis=0, keepdims=True)
    epoch_bl = epoch_csd - bl

    if cond == "AAAA":
        aaaa_p4_epochs.append(epoch_bl)
    else:
        omission_epochs[cond].append(epoch_bl)

print("Epoch counts per condition from continuous run:")
print(f"  AAAA (p4 ref): {len(aaaa_p4_epochs)}")
for c in ["AAAX","AAXA","AXAA"]:
    print(f"  {c}: {len(omission_epochs[c])}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes = axes.flatten()

ref_mean = np.stack(aaaa_p4_epochs).mean(axis=0) if aaaa_p4_epochs else None

for ax_i, cond in enumerate(["AAAA","AAAX","AAXA","AXAA"]):
    ax = axes[ax_i]
    if cond == "AAAA":
        epochs = aaaa_p4_epochs
        label = "p4 reference (AAAA)"
    else:
        epochs = omission_epochs[cond]
        label = f"omission epochs ({cond})"

    if not epochs:
        ax.text(0.5, 0.5, "no epochs", ha="center", va="center"); continue

    epoch_arr = np.stack(epochs)           # (n_ep, WIN_LEN_CT, n_contacts)
    mean_ep   = epoch_arr.mean(axis=0)    # (WIN_LEN_CT, n_contacts)

    # Show each contact as a separate trace
    for ci in range(N_CONTACTS):
        alpha = 0.4 if N_CONTACTS > 8 else 0.7
        ax.plot(t_local_ct, mean_ep[:, ci], color=contact_colors[ci], lw=0.8, alpha=alpha)

    if cond != "AAAA" and ref_mean is not None:
        # Overlay AAAA reference for comparison (deep contact only)
        ax.plot(t_local_ct, ref_mean[:, N_CONTACTS//2],
                "k--", lw=1.2, alpha=0.5, label="AAAA ref (mid)")

    ax.axvline(0, color="red", lw=1.2, ls="--")
    ax.axvspan(-BL_PRE_MS, -BL_POST_MS, color="gray", alpha=0.1, label="baseline")
    ax.set_title(f"{label}\n({len(epochs)} epochs)", fontsize=9)
    ax.set_xlabel("ms from expected onset"); ax.set_ylabel("CSD-proxy (a.u.)")
    ax.legend(fontsize=6); ax.grid(True, alpha=0.2)

fig.suptitle("Omission-locked CSD-proxy (continuous run, baseline-corrected)\n"
              f"proxy scaffold — homeostasis+plasticity active",
              fontsize=10)
fig.savefig(OUTPUT_DIR / "continuous_omission_locked.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved: continuous_omission_locked.png  (proxy scaffold)")


## 11 · Manifest & validation report

In [ ]:
manifest = {
    "notebook": "jaxfne_v040_continuous_omission_oddball",
    "jaxfne_version": jtfne.__version__,
    "paradigm": {
        "name": "continuous_sequential_4item_omission",
        "conditions": CONDITIONS,
        "condition_probs": COND_PROBS,
        "slot_timing_ms": {"p1": P1_ONSET_MS, "p2": P2_ONSET_MS,
                            "p3": P3_ONSET_MS, "p4": P4_ONSET_MS},
        "p_dur_ms": P_DUR_MS, "d_dur_ms": D_DUR_MS,
        "fixation_ms": FX_MS, "total_trial_ms": TOTAL_MS,
        "stimulus_target": "L4_E_neurons",
        "stimulus_amplitude": A_AMP,
        "n_stimulus_targets": len(l4e_idx),
    },
    "config": {
        "N_NEURONS": N_NEURONS, "N_TRIALS": N_TRIALS, "DT_MS": DT_MS, "SEED": SEED,
        "N_CONTACTS": N_CONTACTS, "N_SEQ_CONTINUOUS": N_SEQ,
        "homeostasis_active": True, "plasticity_active": True,
        "homeostasis_params": HOMEO_PARAMS,
    },
    "spectrolaminar_similarity_pct": sim_scores,
    "continuous_omission_epochs": {
        "AAAA_p4_ref": len(aaaa_p4_epochs),
        **{c: len(omission_epochs[c]) for c in ["AAAX","AAXA","AXAA"]},
    },
    "claim_level": "computational_scaffold",
    "field_solver_status": "linear_solver",
    "physical_amplitude_calibrated": False,
    "task_paradigm_source": "01_task_paradigm_timing_data.md",
}

with open(OUTPUT_DIR / "omission_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

val = {
    "all_spikes_finite": all(np.isfinite(results[c]["spk"]).all() for c in CONDITIONS),
    "all_csd_finite":    all(np.isfinite(results[c]["csd"]).all() for c in CONDITIONS),
    "continuous_csd_finite": bool(np.isfinite(csd_cont).all()),
    "similarity_in_range": all(0<=v<=100 for v in sim_scores.values()),
    "all_conditions_simulated": len(results) == len(CONDITIONS),
    "n_seq_continuous": N_SEQ,
    "omission_epochs_found": len(omission_onsets_cont),
}
with open(OUTPUT_DIR / "omission_validation.json", "w") as f:
    json.dump(val, f, indent=2)

print("omission_manifest.json + omission_validation.json written.")
print("Validation:", val)
print("\nSimilarity scores:")
for c, s in sim_scores.items():
    print(f"  {c}: {s:.1f}%")
